In [ ]:
import os
from pathlib import Path
import pandas as pd
import time
from datetime import datetime
from sklearn.linear_model import LinearRegression
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
from dotenv import load_dotenv

from src.analyse.utils import get_straight_line_distance, wrangle_actuals

load_dotenv()

In [ ]:
calculate_distance = False
save_figs = False

In [ ]:
today = pd.Timestamp.now(tz="UTC")
today

In [ ]:
output_dir = Path(f"outputs/nb03_analyse_listings_{datetime.now().strftime('%Y%m%d-%H%M%S')}")
if save_figs:
    os.makedirs(output_dir, exist_ok=True)

In [ ]:
input_dir = Path('outputs/funda/')
files = sorted(list(input_dir.glob('*.csv')))
files.remove(input_dir / 'extra_info.csv')
files

In [ ]:
df_extra = pd.read_csv(input_dir / 'extra_info.csv')
df_extra.head()

In [ ]:
ind = -1
df_raw = pd.read_csv(files[ind])
df_raw.head()

In [ ]:
df_raw.dtypes

In [ ]:
df = df_raw.copy()
df["publish_date"] = pd.to_datetime(df_raw["publish_date"], format="mixed", utc=True)
df["year_quarter"] = df["publish_date"].dt.to_period("Q")
df["year_month"] = df["publish_date"].dt.to_period("M")
df["price_per_sqm"] = df["price"] / df["floor_area"]
df["status"] = df["status"].apply(lambda x: x if x not in ["none"] else "available")
df["time_on_market_days"] = (today - df["publish_date"]).dt.days
df["time_on_market_days"] = df["time_on_market_days"].where(df["status"] == "available", None)
df["time_on_market_months"] = df["time_on_market_days"] / 30.44
df.sort_values("publish_date", inplace=True)
df.reset_index(drop=True, inplace=True)
df.head()

In [ ]:
df = pd.merge(df, df_extra, on="address", how="left")
df.head()

In [ ]:
df["extra_score"] = df[df_extra.columns.drop("address")].sum(axis=1)
df["workload_score"] = df[["floor_done", "bathroom_renovated", "kitchen_renovated"]].sum(axis=1)
df.tail()

In [ ]:
df_actuals = None
try:
    df_actuals = pd.read_csv(input_dir / "actuals_extracted.csv")
except FileNotFoundError:
    print("Actuals file not found. Please provide the file or check the path.")

df_actuals.head()

In [ ]:
if df_actuals is not None:
    df_actuals = wrangle_actuals(df_actuals)
    display(df_actuals.head())

In [ ]:
df_dists = None

In [ ]:
reference_address = os.getenv("REFERENCE_ADDRESS", "Amsterdam Central Station")
if calculate_distance:
    print(f"Reference address: {reference_address}")

    distances = []
    for ind, row in df.iterrows():
        print(f"row {ind}", end="\r")
        address = row["address"]
        dist = get_straight_line_distance(address, reference_address)
        dist["id"] = row["id"]
        distances.append(dist)

        time.sleep(0.2)

    df_dists = pd.DataFrame(distances)
    display(df_dists.head())

    df = pd.merge(df, df_dists, on="id")
    df.head()


In [ ]:
if df_dists is not None and save_figs:
    df_dists.to_csv(output_dir / f"distances_from_{reference_address.replace(' ', '_')}.csv")
    print("Saved dists dataframe")
elif df_dists is None:
    print("Try reading the distances dataframe from a file.")
    try:
        file_name = f"distances_from_{reference_address.replace(' ', '_')}.csv"
        df_dists = pd.read_csv(input_dir / file_name)
        print(f"Distances dataframe loaded from file {file_name}.")
        df = pd.merge(df, df_dists, on="id")
        print("Merged distances into main dataframe.")
    except FileNotFoundError:
        print("Distances file not found. Please calculate distances or provide the file.")

In [ ]:
df_grouped = df.groupby("year_month").agg(
    num_listings=("id", "count"),
    price_avg=("price", "mean"),
    price_median=("price", "median"),
    price_per_sqm=("price_per_sqm", "mean"),
    living_area=("floor_area", "mean"),
)
df_grouped.reset_index(inplace=True)
df_grouped["year_month"] = df_grouped["year_month"].dt.to_timestamp()
df_grouped.sort_values("year_month", inplace=True)
df_grouped.dtypes

In [ ]:
_ = plt.figure(figsize=(12, 6))

sns.violinplot(
    data=df,
    x="year_quarter",
    y="price_per_sqm",
    inner="quartile",
)

plt.axhline(
    y=df["price_per_sqm"].mean(),
    xmin=-0.5,
    xmax=len(df_grouped) - 0.5,
    color="red",
    linestyle="dashed",
    label="Overall Average",
)

plt.xticks(rotation=45, ha="right")
plt.legend()
plt.title("Price per sqm by Year Quarter")
plt.xlabel("Year Quarter")
plt.ylabel("Price per sqm")

if save_figs:
    plt.savefig(output_dir / "price_per_sqm_violinplot.png", bbox_inches="tight")

plt.show()

In [ ]:
def plot_price_per_sqm_over_time(df):
    _ = plt.figure(figsize=(12, 6))

    sns.boxplot(
        data=df,
        x="year_month",
        y="price_per_sqm",
    )

    avg = df["price_per_sqm"].mean()
    plt.axhline(
        y=avg,
        color="red",
        linestyle="dashed",
        label=f"Overall Average ({avg:,.0f} €/m²)",
    )

    plt.title("Average Price per Square Meter Over Time")
    plt.xticks(rotation=45, ha="right")
    plt.legend()

    if save_figs:
        plt.savefig(output_dir / "price_per_sqm_over_time.png", bbox_inches="tight")

    plt.show()

In [ ]:
plot_price_per_sqm_over_time(df)

In [ ]:
df.head()

In [ ]:
df.status.unique()

In [ ]:
def filter_df(df, match_criteria):
    df_filtered = df.copy()
    for col, criteria in match_criteria.items():
        if isinstance(criteria, tuple) and len(criteria) == 2 and all(isinstance(c, (int, float)) for c in criteria):
            df_filtered = df_filtered[(df_filtered[col] >= criteria[0]) & (df_filtered[col] <= criteria[1])]
        elif isinstance(criteria, tuple):
            df_filtered = df_filtered[df_filtered[col].isin(criteria)]
    return df_filtered

In [ ]:
match_criteria = {
    "floor_area": (50, 100),
    "number_of_bedrooms": (1, 4),
    # "energy_label": ("D", "C", "B", "A", "A+", "A++"),
    "status": ("available", "under_bid", "sold_under_reservation", "sold"),
    # "status": ("available", "under_bid"),
}

df_filtered = filter_df(df, match_criteria)
df_filtered.head()

In [ ]:
plot_price_per_sqm_over_time(df_filtered)

In [ ]:
cols_of_interest = [
    "address", "price_per_sqm",
    "price", "floor_area", "number_of_bedrooms",
    "distance_km", "estimated_walking_min",
    "energy_label", "status", "url", "publish_date",
    "time_on_market_months", "extra_score", "workload_score"
]
cols_not_available = [col for col in cols_of_interest if col not in df_filtered.columns]
for col in cols_not_available:
    cols_of_interest.remove(col)
    print(f"Column '{col}' not found in dataframe, skipping it.")

df_filtered[cols_of_interest]

In [ ]:
df_filtered[cols_of_interest].sort_values("price_per_sqm")

In [ ]:
criteria_available = {
    "status": ("available", "under_bid"),
}
df_available = filter_df(df_filtered, criteria_available)
df_available

In [ ]:
criteria = {
    "status": ("available", "under_bid"),
    "price": (0, 400000),
    "floor_area": (75, 110),
}
df_available = filter_df(df, criteria)
if save_figs:
    df_available.to_csv(output_dir / "df_available.csv", index=False)
    print("Saved available listings dataframe")
df_available.sort_values("price_per_sqm")

In [ ]:
def plot_price_per_sqm_floor_area_scatter(df, size_col="extra_score", hue_col="workload_score", palette=None, title="Price per Square Meter vs. Floor Area", save_figs=False):
    fig = plt.figure(figsize=(8,8))
    sns.scatterplot(
        data=df,
        x="floor_area",
        y="price_per_sqm",
        size=size_col,
        hue=hue_col,
        palette=palette
    )
    floor_avg = df["floor_area"].mean()
    plt.axvline(
        x=floor_avg,
        label=f"avg floor area ({floor_avg:,.0f})"
    )
    price_sqm_avg = df["price_per_sqm"].mean()
    plt.axhline(
        price_sqm_avg, label=f"avg price per sqm ({price_sqm_avg:,.0f})"
    )

    plt.grid()
    plt.legend()
    plt.title(title)
    plt.legend(loc="upper right", bbox_to_anchor=(1.4, 1))
    if save_figs:
        plt.savefig(output_dir / "price_per_sqm_vs_floor_area.png", bbox_inches="tight")
    plt.show()

In [ ]:
plot_price_per_sqm_floor_area_scatter(df_available, title="Price per Square Meter vs. Floor Area For Available & Under Bid", save_figs=save_figs)

In [ ]:
criteria = {
    "status": ("under_bid", "sold_under_reservation", "sold"),
    "floor_area": (75, 110),
}
df_sold = filter_df(df, criteria)
plot_price_per_sqm_floor_area_scatter(
    df_sold,
    size_col="estimated_walking_min",
    hue_col="year_month",
    palette="viridis",
    title="Price per Square Meter vs. Floor Area For Sold & Under Reservation", 
    save_figs=save_figs
)

In [ ]:
if df_actuals is not None:
    plot_price_per_sqm_floor_area_scatter(
        df_actuals,
        size_col="price",
        hue_col="year_quarter",
        palette="viridis",
        title="Price per Square Meter vs. Floor Area For Actual Sales",
        save_figs=save_figs
    )

In [ ]:
_ = plt.figure(figsize=(10,10))
sns.scatterplot(
    data=df,
    x="floor_area",
    y="price_per_sqm",
    hue="year_month",
    palette="viridis",
)

plt.grid()
plt.title("Price per sqm by Floor Area and Year Month")
plt.xlabel("Floor Area (m²)")
plt.ylabel("Price per sqm (€)")

if save_figs:
    plt.savefig(output_dir / "price_per_sqm_by_floor_area_and_year_month.png", bbox_inches="tight")

## LR to predict price per sqm in the area of actuals

In [ ]:
def normalize_dates_for_lr(df, date_col="date"):
    X = df[date_col].astype(np.int64).values.reshape(-1, 1)
    X = X / 1e9 / 86400 / 30  # Convert to months
    X = X - 2000  # Normalize to start from year 2000
    return X

def fit_lr_to_predict_price_per_sqm(df):
    # Convert date to numeric (months since epoch)
    X = normalize_dates_for_lr(df, date_col='date')

    y = df['price_per_sqm'].values

    # Fit the model
    model = LinearRegression()
    model.fit(X, y)

    # Print model parameters
    print(f"Coefficient (slope): {model.coef_[0]:.2f} €/m² per month")
    print(f"Intercept: {model.intercept_:.2f} €/m²")
    print(f"R² score: {model.score(X, y):.3f}")
    return model

In [ ]:
def plot_price_per_sqm_over_time(df, model: LinearRegression = None, save_figs=False):
    _ = plt.figure(figsize=(12, 6))

    sns.lineplot(
        data=df,
        x="year_month",
        y="price_per_sqm",
        marker="o",
    )

    avg = df["price_per_sqm"].mean()
    plt.axhline(
        y=avg,
        color="red",
        linestyle="dashed",
        label=f"Overall Average ({avg:,.0f} €/m²)",
    )
    if model is not None:
        X = normalize_dates_for_lr(df, date_col="year_month")
        y_pred = model.predict(X)
        plt.plot(df["year_month"], y_pred, 'r-', linewidth=2, label='Linear Fit')

    plt.title("Price per Square Meter Over Time For Actual Sales")
    plt.ylim(0, plt.ylim()[1])
    plt.xlabel("Year-Month")
    plt.ylabel("Price per Square Meter")
    plt.grid()
    plt.tight_layout()
    plt.title("Average Price per Square Meter Over Time")
    plt.xticks(rotation=45, ha="right")
    plt.legend()

    if save_figs:
        plt.savefig(output_dir / "price_per_sqm_over_time.png", bbox_inches="tight")

    plt.show()

if df_actuals is not None:
    plot_price_per_sqm_over_time(df_actuals, model=fit_lr_to_predict_price_per_sqm(df_actuals), save_figs=save_figs)

In [ ]:
if df_actuals is not None:
    df_actuals_filtered = df_actuals[df_actuals["year_quarter"] > "2015Q4"]
    model = fit_lr_to_predict_price_per_sqm(df_actuals_filtered)
    plot_price_per_sqm_over_time(df_actuals_filtered, model=model, save_figs=save_figs)

In [ ]:
dates_of_interest = ["2025-01", "2025-06", "2025-12", "2026-02", "2026-03", "2026-06", "2026-12", "2029-01"]
if model is not None:
    df_dates_of_interest = pd.DataFrame({"date": pd.to_datetime(dates_of_interest).to_period("M").to_timestamp()})
    X = normalize_dates_for_lr(df_dates_of_interest, date_col="date")
    y_pred = model.predict(X)

    X_today = normalize_dates_for_lr(pd.DataFrame({"date": [today]}), date_col="date")
    y_pred_today = model.predict(X_today)

    plt.figure(figsize=(10, 6))
    plt.plot(df_actuals_filtered["year_month"], df_actuals_filtered["price_per_sqm"], marker="o", label="Actual sold listings")
    plt.plot(df_dates_of_interest["date"], y_pred, linewidth=2, label='Linear Fit')
    plt.axvline(today, color="gray", linestyle="dashed", label=f"Today ({y_pred_today[0]:,.0f} €/m²)")
    plt.xlabel("Date")
    plt.ylabel("Price per Square Meter")
    plt.title("Price per Square Meter Over Time with Future Predictions")
    plt.legend()
    plt.grid()

## Merge actuals and asks

In [ ]:
df_actuals_grouped = df_actuals.groupby(["year_quarter"]).agg(
    price_per_sqm_avg=("price_per_sqm", "mean"),
    num_sales=("date", "count")
).reset_index()
df_actuals_grouped["year"] = df_actuals_grouped["year_quarter"].dt.year

df_actuals_grouped.head()

In [ ]:
df_asked_grouped = df_sold.groupby(["year_quarter"]).agg(
    price_per_sqm_avg=("price_per_sqm", "mean"),
    num_sales=("price", "count")
).reset_index()
df_asked_grouped["year_quarter"] = df_asked_grouped["year_quarter"].dt.to_timestamp()
df_asked_grouped["year"] = df_asked_grouped["year_quarter"].dt.year

df_asked_grouped.head()

In [ ]:
cols_oi = ["year_quarter", "price_per_sqm_avg", "num_sales"]
df_actuals_vs_asked =pd.merge(df_actuals_grouped[cols_oi], df_asked_grouped[cols_oi], on=["year_quarter"], suffixes=("_actuals", "_asked"), how="outer")
df_actuals_vs_asked.tail()

In [ ]:
df_actuals_vs_asked["price_diff"] = df_actuals_vs_asked["price_per_sqm_avg_actuals"] - df_actuals_vs_asked["price_per_sqm_avg_asked"]
df_actuals_vs_asked.tail()

In [ ]:
df_to_plot = df_actuals_vs_asked.dropna(subset=["price_diff"])
sns.barplot(df_to_plot, x="year_quarter", y="price_diff")